## Spark write Hive table

In [1]:
import pyspark
import os
import sys
from pyspark.sql import SparkSession

spark_home = "/home/jovyan/spark-3.4.2-bin-hadoop3"
gravitino_connector_jar = os.getenv('SPARK_CONNECTOR_JAR')
os.environ['HADOOP_USER_NAME']="anonymous"

spark = SparkSession.builder \
    .appName("PySpark SQL Example") \
    .config("spark.plugins", "org.apache.gravitino.spark.connector.plugin.GravitinoSparkPlugin") \
    .config("spark.jars", f"/tmp/gravitino/spark/packages/iceberg-spark-runtime-3.4_2.12-1.6.1.jar,/tmp/gravitino/spark/packages/{gravitino_connector_jar}") \
    .config("spark.sql.gravitino.uri", "http://gravitino:8090") \
    .config("spark.sql.gravitino.metalake", "metalake_demo") \
    .config("spark.sql.gravitino.enableIcebergSupport", "true") \
    .config("spark.sql.catalog.catalog_rest", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.catalog_rest.type", "rest") \
    .config("spark.sql.catalog.catalog_rest.uri", "http://gravitino:9001/iceberg/") \
    .config("spark.locality.wait.node", "0") \
    .config("spark.sql.warehouse.dir", "hdfs://hive:9000/user/hive/warehouse") \
    .enableHiveSupport() \
    .getOrCreate()

In [2]:
spark.sql("use catalog_hive")
spark.sql("show databases").show()

+---------+
|namespace|
+---------+
|  company|
|  default|
|  product|
|    sales|
+---------+



In [3]:
spark.sql("CREATE DATABASE IF NOT EXISTS product;")
spark.sql("USE product;")
spark.sql("CREATE TABLE IF NOT EXISTS employees (id INT, name STRING, age INT) PARTITIONED BY (department STRING) STORED AS PARQUET;")
spark.sql("DESC TABLE EXTENDED employees;").show()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|                  id|                 int|   null|
|                name|              string|   null|
|                 age|                 int|   null|
|          department|              string|   null|
|# Partition Infor...|                    |       |
|          # col_name|           data_type|comment|
|          department|              string|   null|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|                Name|   product.employees|       |
|                Type|             MANAGED|       |
|            Location|hdfs://8a33a7e1e7...|       |
|               Owner|           anonymous|       |
|    Table Properties|[hive.stored-as=P...|       |
+--------------------+--------------------+-------+



In [4]:
spark.sql("INSERT OVERWRITE TABLE employees PARTITION(department='Engineering') VALUES (1, 'John Doe', 30), (2, 'Jane Smith', 28);")
spark.sql("INSERT OVERWRITE TABLE employees PARTITION(department='Marketing') VALUES (3, 'Mike Brown', 32);")
spark.sql("SELECT * from employees").show()

+---+----------+---+-----------+
| id|      name|age| department|
+---+----------+---+-----------+
|  2|Jane Smith| 28|Engineering|
|  1|  John Doe| 30|Engineering|
|  3|Mike Brown| 32|  Marketing|
+---+----------+---+-----------+



## Query the table with Trino

In [5]:
%pip install requests==2.32.3 trino==0.335.0

Note: you may need to restart the kernel to use updated packages.


In [6]:
from trino.dbapi import connect

# Create a Trino connector client
conn = connect(
    host="trino",
    port=8080,
    user="admin",
    catalog="catalog_hive",
    schema="http",
)

trino_client = conn.cursor()

In [7]:
print(trino_client.execute("SELECT * FROM catalog_hive.product.employees WHERE department = 'Engineering'").fetchall())

[[1, 'John Doe', 30, 'Engineering'], [2, 'Jane Smith', 28, 'Engineering']]


## Spark write data with Iceberg REST service

In [13]:
spark.sql("use catalog_rest;")
spark.sql("create database if not exists sales;")
spark.sql("use sales;")
spark.sql("create table if not exists customers (customer_id int, customer_name varchar(100), customer_email varchar(100));")

DataFrame[]

In [14]:
spark.sql("insert into customers (customer_id, customer_name, customer_email) values (11,'Rory Brown','rory@123.com');")
spark.sql("insert into customers (customer_id, customer_name, customer_email) values (12,'Jerry Washington','jerry@dt.com');")
spark.sql("select * from customers").show()

+-----------+----------------+--------------+
|customer_id|   customer_name|customer_email|
+-----------+----------------+--------------+
|         11|      Rory Brown|  rory@123.com|
|         12|Jerry Washington|  jerry@dt.com|
+-----------+----------------+--------------+



## Trino do federation query data with Hive and Iceberg

In [15]:
print(trino_client.execute("select * from catalog_hive.sales.customers union select * from catalog_iceberg.sales.customers").fetchall())

[[12, 'Jerry Washington', 'jerry@dt.com'], [11, 'Rory Brown', 'rory@123.com'], [3, 'Leah Swanson', 'leahswanson1069@protonmail.com'], [10, 'Ronan Joyner', 'ronanjoyner5549@aol.com'], [9, 'Raya Mcguire', 'rayamcguire@hotmail.com'], [6, 'Harriet Best', 'harrietbest2890@icloud.com'], [1, 'Nasim Duke', 'nasimduke@hotmail.net'], [7, 'Erasmus Phelps', 'erasmusphelps9105@protonmail.net'], [2, 'Perry Tyler', 'perrytyler@outlook.com'], [8, 'Lenore Wilder', 'lenorewilder@aol.net'], [4, 'Mia Hahn', 'miahahn@yahoo.edu'], [5, 'Quin Hurst', 'quinhurst5485@google.net']]


## Spark write Iceberg table in Minio

### Restart the notebook to start a new Spark context

In [1]:
import pyspark
import os
import sys
from pyspark.sql import SparkSession

spark_home = "/home/jovyan/spark-3.4.2-bin-hadoop3"
gravitino_connector_jar = os.getenv('SPARK_CONNECTOR_JAR')
os.environ['HADOOP_USER_NAME']="anonymous"

spark = SparkSession.builder \
    .appName("PySpark SQL Example") \
    .config("spark.plugins", "org.apache.gravitino.spark.connector.plugin.GravitinoSparkPlugin") \
    .config("spark.jars", \
            f"/tmp/gravitino/spark/packages/iceberg-spark-runtime-3.4_2.12-1.6.1.jar, \
            /tmp/gravitino/spark/packages/{gravitino_connector_jar}, \
            /tmp/gravitino/spark/packages/iceberg-aws-bundle-1.6.1.jar, \
            /tmp/gravitino/spark/packages/mysql-connector-java-8.0.27.jar") \
    .config("spark.sql.gravitino.uri", "http://gravitino:8090") \
    .config("spark.sql.gravitino.metalake", "metalake_demo") \
    .config("spark.sql.gravitino.enableIcebergSupport", "true") \
    .config("spark.sql.catalog.catalog_rest", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.catalog_rest.type", "rest") \
    .config("spark.sql.catalog.catalog_rest.uri", "http://gravitino:9001/iceberg/") \
    .config("spark.locality.wait.node", "0") \
    .config("spark.sql.warehouse.dir", "hdfs://hive:9000/user/hive/warehouse") \
    .enableHiveSupport() \
    .getOrCreate()

In [2]:
spark.sql("USE catalog_iceberg_s3")
spark.sql("CREATE DATABASE IF NOT EXISTS mydatabase;")
spark.sql("SHOW databases;").show()
spark.sql("USE mydatabase;")

+----------+
| namespace|
+----------+
|mydatabase|
+----------+



DataFrame[]

In [10]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS employee (
    id bigint,
    name string,
    department string, 
    hire_date timestamp) 
    USING iceberg PARTITIONED BY (days(hire_date));
""")
spark.sql("DESC TABLE EXTENDED employee;").show()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|                  id|              bigint|   null|
|                name|              string|   null|
|          department|              string|   null|
|           hire_date|           timestamp|   null|
|                    |                    |       |
|      # Partitioning|                    |       |
|              Part 0|     days(hire_date)|       |
|                    |                    |       |
|  # Metadata Columns|                    |       |
|            _spec_id|                 int|       |
|          _partition|struct<hire_date_...|       |
|               _file|              string|       |
|                _pos|              bigint|       |
|            _deleted|             boolean|       |
|                    |                    |       |
|# Detailed Table ...|                    |       |
|           

In [11]:
spark.sql("""
    INSERT INTO employee
    VALUES
    (1, 'Alice', 'Engineering', TIMESTAMP '2021-01-01 09:00:00'),
    (2, 'Bob', 'Marketing', TIMESTAMP '2021-02-01 10:30:00'),
    (3, 'Charlie', 'Sales', TIMESTAMP '2021-03-01 08:45:00');
""")
spark.sql("SELECT * FROM employee;").show()
spark.sql("SELECT * FROM employee WHERE date(hire_date) = '2021-01-01';").show()

+---+-------+-----------+-------------------+
| id|   name| department|          hire_date|
+---+-------+-----------+-------------------+
|  1|  Alice|Engineering|2021-01-01 09:00:00|
|  2|    Bob|  Marketing|2021-02-01 10:30:00|
|  3|Charlie|      Sales|2021-03-01 08:45:00|
+---+-------+-----------+-------------------+

+---+-----+-----------+-------------------+
| id| name| department|          hire_date|
+---+-----+-----------+-------------------+
|  1|Alice|Engineering|2021-01-01 09:00:00|
+---+-----+-----------+-------------------+



### Operation example: UPDATE, DELETE, MERGE

In [12]:
spark.sql("UPDATE employee SET department = 'Jenny' WHERE id = 1;")
spark.sql("SELECT * FROM employee;").show()
spark.sql("DELETE FROM employee WHERE id < 2;")
spark.sql("SELECT * FROM employee;").show()

+---+-------+----------+-------------------+
| id|   name|department|          hire_date|
+---+-------+----------+-------------------+
|  1|  Alice|     Jenny|2021-01-01 09:00:00|
|  2|    Bob| Marketing|2021-02-01 10:30:00|
|  3|Charlie|     Sales|2021-03-01 08:45:00|
+---+-------+----------+-------------------+

+---+-------+----------+-------------------+
| id|   name|department|          hire_date|
+---+-------+----------+-------------------+
|  2|    Bob| Marketing|2021-02-01 10:30:00|
|  3|Charlie|     Sales|2021-03-01 08:45:00|
+---+-------+----------+-------------------+



In [13]:
spark.sql("""
    MERGE INTO employee
    USING (SELECT 4 as id, 'David' as name, 'Engineering' as department, TIMESTAMP '2021-04-01 09:00:00' as hire_date) as new_employee
    ON employee.id = new_employee.id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *;  
""")
spark.sql("SELECT * FROM employee;").show()

+---+-------+-----------+-------------------+
| id|   name| department|          hire_date|
+---+-------+-----------+-------------------+
|  4|  David|Engineering|2021-04-01 09:00:00|
|  2|    Bob|  Marketing|2021-02-01 10:30:00|
|  3|Charlie|      Sales|2021-03-01 08:45:00|
+---+-------+-----------+-------------------+



In [14]:
spark.sql("""
    MERGE INTO employee
    USING (SELECT 4 as id, 'David' as name, 'Engineering' as department, TIMESTAMP '2021-04-01 09:00:00' as hire_date) as new_employee
    ON employee.id = new_employee.id
    WHEN MATCHED THEN DELETE
    WHEN NOT MATCHED THEN INSERT *; 
""")
spark.sql("SELECT * FROM employee;").show()

+---+-------+----------+-------------------+
| id|   name|department|          hire_date|
+---+-------+----------+-------------------+
|  2|    Bob| Marketing|2021-02-01 10:30:00|
|  3|Charlie|     Sales|2021-03-01 08:45:00|
+---+-------+----------+-------------------+



### Produres example

In [26]:
spark.sql("CALL catalog_iceberg_s3.system.ancestors_of('catalog_iceberg_s3.mydatabase.employee')").show()

+-------------------+-------------+
|        snapshot_id|    timestamp|
+-------------------+-------------+
|2419285818310310315|1758180679672|
|1722595041980053768|1758180675138|
|3103471394041090990|1758180655742|
|5972711573298652625|1758180655325|
| 389876088309693951|1758180635108|
+-------------------+-------------+



In [37]:
# Use the snapshot_id you want in ancestor snapshot_ids above
snapshot_id="389876088309693951"
spark.sql(f"CALL catalog_iceberg_s3.system.rollback_to_snapshot('catalog_iceberg_s3.mydatabase.employee', {snapshot_id})").show()
spark.sql("SELECT * FROM employee;").show()

+--------------------+-------------------+
|previous_snapshot_id|current_snapshot_id|
+--------------------+-------------------+
| 2419285818310310315| 389876088309693951|
+--------------------+-------------------+

+---+-------+-----------+-------------------+
| id|   name| department|          hire_date|
+---+-------+-----------+-------------------+
|  1|  Alice|Engineering|2021-01-01 09:00:00|
|  2|    Bob|  Marketing|2021-02-01 10:30:00|
|  3|Charlie|      Sales|2021-03-01 08:45:00|
+---+-------+-----------+-------------------+



In [38]:
spark.sql("CALL catalog_iceberg_s3.system.set_current_snapshot('catalog_iceberg_s3.mydatabase.employee', 2419285818310310315)").show()
spark.sql("CALL catalog_iceberg_s3.system.ancestors_of('catalog_iceberg_s3.mydatabase.employee')").show()

+--------------------+-------------------+
|previous_snapshot_id|current_snapshot_id|
+--------------------+-------------------+
|  389876088309693951|2419285818310310315|
+--------------------+-------------------+

+-------------------+-------------+
|        snapshot_id|    timestamp|
+-------------------+-------------+
|2419285818310310315|1758180679672|
|1722595041980053768|1758180675138|
|3103471394041090990|1758180655742|
|5972711573298652625|1758180655325|
| 389876088309693951|1758180635108|
+-------------------+-------------+



In [34]:
spark.sql("SELECT * FROM employee TIMESTAMP AS OF '2025-09-18 14:28:00';").show()
spark.sql("SELECT * FROM employee FOR SYSTEM_TIME AS OF '2025-09-18 14:28:00';").show()

+---+-------+----------+-------------------+
| id|   name|department|          hire_date|
+---+-------+----------+-------------------+
|  1|  Alice|     Jenny|2021-01-01 09:00:00|
|  2|    Bob| Marketing|2021-02-01 10:30:00|
|  3|Charlie|     Sales|2021-03-01 08:45:00|
+---+-------+----------+-------------------+

+---+-------+----------+-------------------+
| id|   name|department|          hire_date|
+---+-------+----------+-------------------+
|  1|  Alice|     Jenny|2021-01-01 09:00:00|
|  2|    Bob| Marketing|2021-02-01 10:30:00|
|  3|Charlie|     Sales|2021-03-01 08:45:00|
+---+-------+----------+-------------------+



In [35]:
spark.sql("DESC EXTENDED employee;").show()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|                  id|              bigint|   null|
|                name|              string|   null|
|          department|              string|   null|
|           hire_date|           timestamp|   null|
|                    |                    |       |
|      # Partitioning|                    |       |
|              Part 0|     days(hire_date)|       |
|                    |                    |       |
|  # Metadata Columns|                    |       |
|            _spec_id|                 int|       |
|          _partition|struct<hire_date_...|       |
|               _file|              string|       |
|                _pos|              bigint|       |
|            _deleted|             boolean|       |
|                    |                    |       |
|# Detailed Table ...|                    |       |
|           

## Spark write Paimon table in HDFS

### Restart the notebook to start a new Spark context 

In [1]:
import pyspark
import os
import sys
from pyspark.sql import SparkSession

gravitino_connector_jar = os.getenv('SPARK_CONNECTOR_JAR')
os.environ['HADOOP_USER_NAME']="anonymous"

spark = SparkSession.builder \
    .appName("Spark Paimon Example") \
    .config("spark.plugins", "org.apache.gravitino.spark.connector.plugin.GravitinoSparkPlugin") \
    .config("spark.jars", f"/tmp/gravitino/spark/packages/{gravitino_connector_jar},/tmp/gravitino/spark/packages/paimon-spark-3.4-0.8.2.jar,") \
    .config("spark.sql.gravitino.uri", "http://gravitino:8090") \
    .config("spark.sql.gravitino.metalake", "metalake_demo") \
    .config("spark.sql.extensions", "org.apache.paimon.spark.extensions.PaimonSparkSessionExtensions") \
    .config("spark.locality.wait.node", "0") \
    .config("spark.sql.warehouse.dir", "hdfs://hive:9000/user/hive/warehouse") \
    .config("spark.sql.gravitino.enablePaimonSupport", "true") \
    .enableHiveSupport() \
    .getOrCreate()

In [2]:
spark.sql("use catalog_paimon")
spark.sql("CREATE DATABASE IF NOT EXISTS mydatabase;")
spark.sql("SHOW DATABASES;").show()
spark.sql("USE mydatabase;")

+--------------+
|     namespace|
+--------------+
|access_control|
|       default|
|    mydatabase|
|         sales|
+--------------+



DataFrame[]

In [3]:
spark.sql("drop table if exists employee")

DataFrame[]

In [4]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS employee (
      id BIGINT,
      name STRING,
      department STRING,
      hire_date TIMESTAMP
    )
    USING paimon
    PARTITIONED BY (name)
""")
spark.sql("SHOW TABLES;").show()

+----------+---------+-----------+
| namespace|tableName|isTemporary|
+----------+---------+-----------+
|mydatabase| employee|      false|
+----------+---------+-----------+



In [6]:
spark.sql("""
    INSERT INTO employee
    VALUES
    (1, 'Alice', 'Engineering', TIMESTAMP '2021-01-01 09:00:00'),
    (2, 'Bob', 'Marketing', TIMESTAMP '2021-02-01 10:30:00'),
    (3, 'Charlie', 'Sales', TIMESTAMP '2021-03-01 08:45:00');
""")
spark.sql("SELECT * FROM employee;").show()

+---+-------+-----------+-------------------+
| id|   name| department|          hire_date|
+---+-------+-----------+-------------------+
|  1|  Alice|Engineering|2021-01-01 09:00:00|
|  2|    Bob|  Marketing|2021-02-01 10:30:00|
|  3|Charlie|      Sales|2021-03-01 08:45:00|
+---+-------+-----------+-------------------+



## Spark write Hudi table in HDFS

### Restart the notebook to start a new Spark context 

In [1]:
from pyspark.sql import SparkSession
import os

gravitino_connector_jar = os.getenv('SPARK_CONNECTOR_JAR')
os.environ['HADOOP_USER_NAME']="anonymous"

spark = SparkSession.builder \
    .appName("Hudi Example") \
    .config("spark.plugins", "org.apache.gravitino.spark.connector.plugin.GravitinoSparkPlugin") \
    .config("spark.jars", f"/tmp/gravitino/spark/packages/hudi-spark3.4-bundle_2.12-1.0.2.jar,/tmp/gravitino/spark/packages/gravitino-spark-connector-runtime-3.4_2.12-1.0.0.jar") \
    .config("spark.sql.gravitino.uri", "http://gravitino:8090") \
    .config("spark.sql.gravitino.metalake", "metalake_demo") \
    .config("spark.locality.wait.node", "0") \
    .config("spark.sql.warehouse.dir", "hdfs://hive:9000/user/hive/warehouse") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .getOrCreate()

In [2]:
%pip install hdfs

Note: you may need to restart the kernel to use updated packages.


In [3]:
from hdfs import InsecureClient
import os

# Create a HDFS connector client
hdfs_client = InsecureClient("http://hive:50070", user='anonymous')

# List HDFS file and directories
print(hdfs_client.list('/user/hive/warehouse'))

[]


In [4]:
data = [("1", "book", 10.0, "2025-09-11 09:30:00"),
        ("2", "pen", 2.0, "2025-09-11 09:31:00")]

df = spark.createDataFrame(data, ["id", "name", "price", "ts"])

hudi_options = {
    "hoodie.table.name": "hudi_table_cow",
    "hoodie.datasource.write.recordkey.field": "id",
    "hoodie.datasource.write.precombine.field": "ts",
    "hoodie.datasource.write.operation": "insert",
    "hoodie.datasource.write.table.type": "COPY_ON_WRITE",
}

df.write.format("hudi") \
  .option("hoodie.table.name", "hudi_table_cow") \
  .option("hoodie.datasource.write.recordkey.field", "id") \
  .option("hoodie.datasource.write.precombine.field", "ts") \
  .option("hoodie.datasource.write.operation", "upsert") \
  .option("hoodie.datasource.write.table.type", "COPY_ON_WRITE") \
  .option("hoodie.datasource.hive_sync.enable", "true") \
  .option("hoodie.datasource.hive_sync.database", "hudi") \
  .option("hoodie.datasource.hive_sync.table", "hudi_table_cow") \
  .option("hoodie.datasource.hive_sync.mode", "hms") \
  .option("hoodie.datasource.hive_sync.metastore.uris", "thrift://hive:9083") \
  .mode("overwrite") \
  .save("hdfs://hive:9000/user/hive/warehouse/hudi.db/hudi_table_cow")

In [6]:
print(hdfs_client.list('/user/hive/warehouse'))
print(hdfs_client.list('/user/hive/warehouse/hudi.db'))

['hudi.db']
['hudi_table_cow']
